# Feature Engineering


## Library import and settings

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', '{:.4f}'.format)


## Load the Dataset

In [2]:
listings = pd.read_parquet("../data/processed/listings_clean.parquet")


## Shortly reminder


In [3]:
listings.info()


<class 'pandas.DataFrame'>
RangeIndex: 2852 entries, 0 to 2851
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              2852 non-null   int64         
 1   name                            2852 non-null   str           
 2   host_id                         2852 non-null   int64         
 3   host_name                       2849 non-null   str           
 4   neighbourhood                   2852 non-null   int64         
 5   latitude                        2852 non-null   float64       
 6   longitude                       2852 non-null   float64       
 7   room_type                       2852 non-null   str           
 8   price                           2537 non-null   float64       
 9   minimum_nights                  2852 non-null   int64         
 10  number_of_reviews               2852 non-null   int64         
 11  last_review    

## Add Derived Features

### Useful statistics

In [4]:
Q1 = listings["price"].quantile(0.25)
Q3 = listings["price"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


### Feature `has_last_review`

In [5]:
listings["has_last_review"] = listings["last_review"].notna()


### Feature `has_price`

In [6]:
listings["has_price"] = listings["price"].notna()


### Feature `is_price_outlier`

In [7]:
listings["is_price_outlier"] = listings["price"] > upper_bound


### Feature `host_size_segment`

In [8]:
bins = [1, 2, 5, 999999]
labels = ["Individual host", "Experienced host", "Industrial host"]

listings["host_size_segment"] = pd.cut(listings["calculated_host_listings_count"], bins=bins, labels=labels, right=False)


### Feature `availability_ratio`

In [9]:
listings["availability_ratio"] = listings["availability_365"] / 365


### Feature `review_intensity`

In [10]:
listings["review_intensity"] = np.where(
    listings["number_of_reviews"] == 0,
    0,
    listings["number_of_reviews_ltm"] / listings["number_of_reviews"]
)


### Feature `log_price`

In [11]:
listings["log_price"] = np.log1p(listings["price"])


## Save changes

In [12]:
listings.to_parquet("../data/processed/listings_analytics.parquet", index=False)


##